# Runs Tests: Randomness and Two-Sample Mixing

**STA 507 1.0 Nonparametric Statistics** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta507.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [What a run is](#theory-01)
- [Defining the labels](#theory-02)
- [Conditional null and statistic](#theory-03)
- [Exact and approximate inference](#theory-04)
- [Two-sample runs test](#theory-05)
- [Ties and conclusions](#theory-06)

## Detailed theory · Runs tests for ordering and two-sample mixing

<a id="theory-01"></a>

### What a run is
A run is a maximal uninterrupted sequence of the same label. The sequence ++−−+ has three runs. Original order is the data: sorting it destroys the question. A runs test investigates ordering rather than the marginal number of successes; a binomial fairness test cannot substitute for it.

<a id="theory-02"></a>

### Defining the labels
Labels can be naturally binary or defined relative to a stated threshold, often the sample median. If using a median, declare what happens to equal values. Removing ties changes the sequence and effective counts. Do not alternate an arbitrary tie-breaking rule between analyses.

<a id="theory-03"></a>

### Conditional null and statistic
Under random ordering conditional on $n_1$ and $n_2$, every arrangement of those counts is equally likely. The statistic R is one plus the number of adjacent changes. Small R suggests grouping or persistence; large R suggests excessive alternation.
$$E[R]=1+\frac{2n_1n_2}{n_1+n_2},$$
$$Var(R)=\frac{2n_1n_2(2n_1n_2-n_1-n_2)}
{(n_1+n_2)^2(n_1+n_2-1)}.$$
If one label never occurs, usual two-label approximation formulas are not useful for a randomness claim.

<a id="theory-04"></a>

### Exact and approximate inference
For small counts, enumerate or use a matching exact reference for R. A two-sided definition must be stated because the null is discrete; lower/upper-tail allocation and distance-from-mean rules may differ. With adequate counts and nonzero variance, use $Z=(R-E[R])/\sqrt{Var(R)}$, declaring a continuity correction if applied. A normal approximation is not made exact by reporting many digits.

<a id="theory-05"></a>

### Two-sample runs test
To compare two independent continuous samples, pool values and sort by the measured variable, then write the group label at each pooled position. Few runs mean the group labels segregate in value order, suggesting unequal distributions. This is the Wald–Wolfowitz two-sample runs idea. Here sorting is intentional because the ordering variable defines the two-sample construction; it is different from a one-sample time-order runs test.

<a id="theory-06"></a>

### Ties and conclusions
Tied pooled values make label order ambiguous. Use a specified tie-aware method or justified permutation calibration rather than an order chosen to create significance. Report sample sizes, labelling rule, R, null reference and tail. Non-rejection means insufficient ordering evidence; it does not establish independent randomness of the underlying process.

## Concepts covered

**Source alignment:** Handout 1: single-sample runs test for randomness (week 4). Handout 4: Single Sample Runs Test.

**Mission:** Match each design and measurement scale to a test, then explain the result in context.

## Worked calculation · Runs test: label the runs before calculating

### Question and assumptions

The ordered sequence is +,+,−,−,+,−,+,+,−,−. Test random ordering of the two labels, conditional on their counts. The labels are already defined; no median ties need removing.

### Procedure

1. Keep the original order; never sort a runs-test sequence.
2. Start run 1 at the first value. Add one whenever the label changes.
3. Count n+ and n−, then calculate E(R) and Var(R).
4. For this small example enumerate all arrangements with the same label counts for an exact conditional test.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Position | Label | New run? | Run number |
| --- | --- | --- | --- |
| 1 | + | yes | 1 |
| 2 | + | no | 1 |
| 3 | - | yes | 2 |
| 4 | - | no | 2 |
| 5 | + | yes | 3 |
| 6 | - | yes | 4 |
| 7 | + | yes | 5 |
| 8 | + | no | 5 |
| 9 | - | yes | 6 |
| 10 | - | no | 6 |

| Quantity | Value |
| --- | --- |
| n+ | 5 |
| n− | 5 |
| Observed R | 6 |
| E(R) | 6 |
| Var(R) | 2.22222 |
| Exact distance-from-mean two-sided p | 1 |
| Decision at .05 | Do not reject randomness |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


sequence=list("++--+-++--"); run=0; previous=None; rows=[]
for i,label in enumerate(sequence,1):
    new=label!=previous; run+=new; rows.append([i,label,"yes" if new else "no",run]); previous=label
show_table(["Position","Label","New run?","Run number"],rows)
n1=sequence.count("+"); n2=sequence.count("-"); n=n1+n2
ER=1+2*n1*n2/n; VR=2*n1*n2*(2*n1*n2-n)/((n**2)*(n-1))
def count_runs(s):return 1+sum(s[i]!=s[i-1] for i in range(1,len(s)))
null=[]
for locations in combinations(range(n),n1):
    loc=set(locations); null.append(count_runs(["+" if i in loc else "-" for i in range(n)]))
p=sum(abs(r-ER)>=abs(run-ER)-1e-12 for r in null)/len(null)
show_table(["Quantity","Value"],[["n+",n1],["n−",n2],["Observed R",run],["E(R)",ER],["Var(R)",VR],["Exact distance-from-mean two-sided p",p],["Decision at .05","Reject randomness" if p<=.05 else "Do not reject randomness"]])
assert run==6 and abs(ER-6)<1e-12


### Decision and interpretation

This example has the expected number of runs. Too few runs suggest clustering; too many suggest excessive alternation. The exact two-sided convention here sums arrangements whose distance from E(R) is at least the observed distance; other two-sided conventions must be named if used.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(507)

In [ ]:
seq=list('HHHTTTHHTTHTHTHTTTHH')
runs=1+sum(seq[i]!=seq[i-1] for i in range(1,len(seq)))
n1=seq.count('H'); n2=seq.count('T')
ER=1+2*n1*n2/(n1+n2)
VR=2*n1*n2*(2*n1*n2-n1-n2)/(((n1+n2)**2)*(n1+n2-1))
print('n1,n2,r:',n1,n2,runs,'E(R):',round(ER,3),'Var(R):',round(VR,3))
print('handout critical bounds at n1=n2=10, alpha=.05: 6 and 16')

### Practice · Find the runs
1. Count runs in A,A,B,B,A,B,B,B,A.
2. Give one pattern that creates too few runs and one that creates too many.
3. The source’s coin sequence has n₁=n₂=10 and r=11, with critical bounds 6 and 16. State the decision.

<details><summary>Answer sketches</summary>
1. AA | BB | A | B | BB | A: six runs. 2. Long clusters create too few; strict alternation creates too many. 3. 6<11<16, so do not reject randomness at 5% using those bounds.
</details>